In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
import json
import os

# Chargement des interactions
train_interactions = pd.read_csv(
    "../data/processed/train_interactions.csv"
)

# Chargement du catalogue
news_catalog = pd.read_csv(
    "../data/processed/news_catalog.csv"
)

# Chargement des impressions dev
dev_impressions = pd.read_csv(
    "../data/processed/dev_impressions.csv.gz"
)

# Chargement de l'échantillon commun
sample_impressions = pd.read_csv(
    "../data/processed/evaluation_sample_1000.csv"
)["impression_id"].tolist()

print("Interactions train :", len(train_interactions))
print("Articles :", len(news_catalog))
print("Impressions dev :", dev_impressions["impression_id"].nunique())
print("Impressions de l'échantillon :", len(sample_impressions))

Interactions train : 234468
Articles : 65238
Impressions dev : 73152
Impressions de l'échantillon : 1000


In [2]:
# Popularité de chaque article à partir des interactions TRAIN uniquement

# Nombre d'utilisateurs ayant cliqué sur chaque article
popularity_scores = (
    train_interactions
    .groupby("news_id")["user_id"]
    .nunique()
    .to_dict()
)

# Ajouter le score de popularité au catalogue complet
news_popularity = news_catalog[["news_id", "category", "title"]].copy()

news_popularity["popularity_score"] = (
    news_popularity["news_id"]
    .map(popularity_scores)
    .fillna(0)
    .astype(int)
)

# Trier les articles du plus populaire au moins populaire
news_popularity = news_popularity.sort_values(
    by="popularity_score",
    ascending=False
).reset_index(drop=True)

print("Nombre total d'articles :", len(news_popularity))
print("Articles ayant au moins un clic :", 
      (news_popularity["popularity_score"] > 0).sum())
print("Articles sans clic :", 
      (news_popularity["popularity_score"] == 0).sum())

display(news_popularity.head(10))

Nombre total d'articles : 65238
Articles ayant au moins un clic : 7713
Articles sans clic : 57525


,news_id,category,title,popularity_score
0,N55689,sports,"Charles Rogers, former Michigan State football...",4257
1,N35729,news,Porsche launches into second story of New Jers...,3281
2,N33619,news,College gymnast dies following training accide...,3214
3,N53585,tv,"Rip Taylor's Cause of Death Revealed, Memorial...",2824
4,N63970,finance,Dean Foods files for bankruptcy,2550
5,N49685,music,Broadway Star Laurel Griggs Suffered Asthma At...,2273
6,N49279,music,Broadway Actress Laurel Griggs Dies at Age 13,2236
7,N287,news,Three school workers charged in death of speci...,2111
8,N23446,lifestyle,Prince Harry and Meghan Markle just shared a n...,1907
9,N51048,news,Rep. Tim Ryan endorses Biden in Democratic pri...,1861


In [3]:
# Calcul de la popularité à partir des interactions train
popularity_df = (
    train_interactions
    .groupby("news_id")
    .size()
    .reset_index(name="popularity_score")
)

# Ajouter les informations des articles
popularity_df = popularity_df.merge(
    news_catalog[["news_id", "category", "title"]],
    on="news_id",
    how="right"
)

# Les articles sans clic dans train ont une popularité égale à 0
popularity_df["popularity_score"] = (
    popularity_df["popularity_score"]
    .fillna(0)
    .astype(int)
)

# Trier par popularité décroissante
popularity_df = popularity_df.sort_values(
    by="popularity_score",
    ascending=False
).reset_index(drop=True)

print("Nombre total d'articles :", len(popularity_df))
print("Articles ayant au moins un clic :",
      (popularity_df["popularity_score"] > 0).sum())
print("Articles sans clic :",
      (popularity_df["popularity_score"] == 0).sum())

display(popularity_df.head(10))

Nombre total d'articles : 65238
Articles ayant au moins un clic : 7713
Articles sans clic : 57525


,news_id,popularity_score,category,title
0,N55689,4257,sports,"Charles Rogers, former Michigan State football..."
1,N35729,3281,news,Porsche launches into second story of New Jers...
2,N33619,3214,news,College gymnast dies following training accide...
3,N53585,2824,tv,"Rip Taylor's Cause of Death Revealed, Memorial..."
4,N63970,2550,finance,Dean Foods files for bankruptcy
5,N49685,2273,music,Broadway Star Laurel Griggs Suffered Asthma At...
6,N49279,2236,music,Broadway Actress Laurel Griggs Dies at Age 13
7,N287,2111,news,Three school workers charged in death of speci...
8,N23446,1907,lifestyle,Prince Harry and Meghan Markle just shared a n...
9,N51048,1861,news,Rep. Tim Ryan endorses Biden in Democratic pri...


In [4]:
popularity_df.to_csv(
    "../data/processed/popularity_scores.csv",
    index=False
)

print("Classement de popularité sauvegardé avec succès.")

Classement de popularité sauvegardé avec succès.


In [5]:
# Charger les historiques de navigation du jeu dev
import pickle

with open("../data/processed/dev_history.pkl", "rb") as f:
    dev_history = pickle.load(f)

# Vérifier le contenu
print("Type :", type(dev_history))
print("Nombre d'historiques :", len(dev_history))

display(dev_history.head())

Type : <class 'pandas.DataFrame'>
Nombre d'historiques : 73152


,impression_id,user_id,history,history_list,history_length,history_indices
0,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,"[N55189, N46039, N51741, N53234, N11276, N264,...",15,"[6892, 30443, 14245, 29198, 28698, 16460, 1609..."
1,2,U60458,N58715 N32109 N51180 N33438 N54827 N28488 N611...,"[N58715, N32109, N51180, N33438, N54827, N2848...",13,"[28645, 18332, 23489, 19314, 17437, 25980, 145..."
2,3,U44190,N56253 N1150 N55189 N16233 N61704 N51706 N5303...,"[N56253, N1150, N55189, N16233, N61704, N51706...",9,"[15207, 17638, 6892, 11138, 31407, 17205, 2360..."
3,4,U87380,N63554 N49153 N28678 N23232 N43369 N58518 N444...,"[N63554, N49153, N28678, N23232, N43369, N5851...",19,"[4963, 31496, 17537, 15839, 23062, 11229, 2771..."
4,5,U9444,N51692 N18285 N26015 N22679 N55556,"[N51692, N18285, N26015, N22679, N55556]",5,"[24779, 28018, 31507, 111, 21109]"


In [6]:
# Créer un dictionnaire :
# impression_id -> ensemble des indices des articles déjà lus

history_lookup = {}

for _, row in dev_history.iterrows():
    history_lookup[row["impression_id"]] = set(
        row["history_indices"]
    )

# Créer un dictionnaire :
# news_id -> score de popularité

popularity_lookup = dict(
    zip(
        popularity_df["news_id"],
        popularity_df["popularity_score"]
    )
)

print("Nombre d'historiques :", len(history_lookup))
print("Nombre d'articles avec un score :", len(popularity_lookup))

Nombre d'historiques : 73152
Nombre d'articles avec un score : 65238


In [7]:
def recommend_popularity(
    impression_df,
    history_lookup,
    popularity_lookup,
    item_to_idx,
    k=10
):
    impression_id = impression_df["impression_id"].iloc[0]

    # Historique de l'utilisateur pour cette impression
    history_indices = history_lookup.get(impression_id, set())

    # Récupérer les articles candidats
    candidates = impression_df.copy()

    # Exclure les articles déjà consultés
    candidates["item_idx"] = candidates["news_id"].map(item_to_idx)

    candidates = candidates[
        ~candidates["item_idx"].isin(history_indices)
    ].copy()

    # Attribuer un score de popularité
    candidates["popularity_score"] = (
        candidates["news_id"]
        .map(popularity_lookup)
        .fillna(0)
    )

    # Trier par popularité décroissante
    candidates = candidates.sort_values(
        by="popularity_score",
        ascending=False,
        kind="stable"
    )

    return candidates.head(k)

In [9]:
import json

with open("../data/processed/mappings.json", "r") as f:
    mappings = json.load(f)

item_to_idx = mappings["item_to_idx"]

# Vérification
print("Nombre d'articles dans le mapping :", len(item_to_idx))
print("Exemple :", list(item_to_idx.items())[:5])

Nombre d'articles dans le mapping : 65238
Exemple : [('N55528', 0), ('N19639', 1), ('N61837', 2), ('N53526', 3), ('N38324', 4)]


In [10]:
test_id = sample_impressions[0]

test_impression = dev_impressions[
    dev_impressions["impression_id"] == test_id
]

test_recommendations = recommend_popularity(
    test_impression,
    history_lookup,
    popularity_lookup,
    item_to_idx,
    k=10
)

display(
    test_recommendations[
        ["news_id", "label", "popularity_score"]
    ]
)

,news_id,label,popularity_score
4,N6916,0,44
18,N11930,0,6
11,N5940,0,3
2,N31958,1,1
0,N28682,0,0
1,N48740,0,0
3,N34130,0,0
5,N5472,0,0
6,N50775,0,0
7,N24802,0,0


In [11]:
import numpy as np

def calculate_metrics(recommended_labels, k=10):

    recommended_labels = np.array(recommended_labels[:k])

    relevant_count = np.sum(recommended_labels)

    precision = relevant_count / k

    # Nombre total d'articles pertinents dans l'impression
    # (calculé séparément lors de l'évaluation)

    return precision


def calculate_recall(recommended_labels, total_relevant):

    if total_relevant == 0:
        return 0.0

    return np.sum(recommended_labels) / total_relevant


def calculate_ndcg(recommended_labels, k=10):

    labels = np.array(recommended_labels[:k])

    if len(labels) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(labels) + 2))

    dcg = np.sum(labels / discounts)

    ideal_labels = np.sort(labels)[::-1]
    idcg = np.sum(ideal_labels / discounts)

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [14]:
results = []

for impression_id in sample_impressions:

    impression_df = dev_impressions[
        dev_impressions["impression_id"] == impression_id
    ].copy()

    if impression_df.empty:
        continue

    # Nombre total d'articles pertinents dans l'impression
    total_relevant = int(impression_df["label"].sum())

    if total_relevant == 0:
        continue

    # Recommandations Popularity
    recommendations = recommend_popularity(
        impression_df,
        history_lookup,
        popularity_lookup,
        item_to_idx,
        k=10
    )

    recommended_labels = recommendations["label"].tolist()

    # Compléter avec des zéros si moins de 10 candidats
    recommended_labels += [0] * (10 - len(recommended_labels))

    precision = calculate_metrics(recommended_labels, k=10)

    recall = calculate_recall(
        recommended_labels,
        total_relevant
    )

    ndcg = calculate_ndcg(
    recommended_labels,
    total_relevant,
    k=10
)

    results.append({
        "impression_id": impression_id,
        "precision": precision,
        "recall": recall,
        "ndcg": ndcg,
        "candidate_count": len(impression_df),
        "relevant_count": total_relevant,
        "recommended_news_ids": recommendations["news_id"].tolist()
    })

popularity_evaluation_df = pd.DataFrame(results)

print("Impressions évaluées :", len(popularity_evaluation_df))
display(popularity_evaluation_df.head())

Impressions évaluées : 1000


,impression_id,precision,recall,ndcg,candidate_count,relevant_count,recommended_news_ids
0,1,0.1,1.0,0.430677,22,1,"[N6916, N11930, N5940, N31958, N28682, N48740,..."
1,2,0.1,1.0,0.630930,7,1,"[N36779, N23513, N20036, N31958, N32536, N4697..."
2,3,0.1,1.0,0.356207,23,1,"[N30290, N6916, N36779, N20036, N11930, N5940,..."
3,4,0.0,0.0,0.000000,26,1,"[N1952, N6916, N11930, N6074, N38620, N6950, N..."
4,5,0.2,1.0,0.455605,14,2,"[N30290, N36779, N23513, N20036, N5940, N31958..."


In [13]:
def calculate_ndcg(recommended_labels, total_relevant, k=10):

    labels = np.array(recommended_labels[:k])

    if len(labels) == 0 or total_relevant == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(labels) + 2))

    # DCG : qualité du classement obtenu
    dcg = np.sum(labels / discounts)

    # IDCG : classement idéal
    ideal_relevant = min(total_relevant, k)

    ideal_labels = np.array(
        [1] * ideal_relevant + [0] * (len(labels) - ideal_relevant)
    )

    idcg = np.sum(ideal_labels / discounts)

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [15]:
popularity_metrics = {
    "Model": "Popularity",
    "Precision@10": popularity_evaluation_df["precision"].mean(),
    "Recall@10": popularity_evaluation_df["recall"].mean(),
    "NDCG@10": popularity_evaluation_df["ndcg"].mean(),
    "Evaluated impressions": len(popularity_evaluation_df),
    "Sample impressions": len(sample_impressions)
}

popularity_metrics_df = pd.DataFrame([popularity_metrics])

display(popularity_metrics_df.round(4))

,Model,Precision@10,Recall@10,NDCG@10,Evaluated impressions,Sample impressions
0,Popularity,0.0693,0.5481,0.3039,1000,1000


In [16]:
# Ensemble des articles recommandés
recommended_articles_popularity = set()

for articles in popularity_evaluation_df["recommended_news_ids"]:
    recommended_articles_popularity.update(articles)

# Calcul de la couverture
popularity_coverage = (
    len(recommended_articles_popularity) / len(news_catalog)
)

print("Articles différents recommandés :",
      len(recommended_articles_popularity))

print("Nombre total d'articles :", len(news_catalog))

print("Coverage :",
      round(popularity_coverage * 100, 2), "%")

Articles différents recommandés : 542
Nombre total d'articles : 65238
Coverage : 0.83 %


In [17]:
# Ensemble des articles recommandés
recommended_articles_popularity = set()

for articles in popularity_evaluation_df["recommended_news_ids"]:
    recommended_articles_popularity.update(articles)

# Calcul de la couverture
popularity_coverage = (
    len(recommended_articles_popularity) / len(news_catalog)
)

print("Articles différents recommandés :",
      len(recommended_articles_popularity))

print("Nombre total d'articles :", len(news_catalog))

print("Coverage :",
      round(popularity_coverage * 100, 2), "%")

Articles différents recommandés : 542
Nombre total d'articles : 65238
Coverage : 0.83 %


In [18]:
# Sauvegarder les résultats détaillés
popularity_evaluation_df.to_csv(
    "../data/processed/popularity_evaluation_details.csv",
    index=False
)

# Ajouter la couverture aux métriques
popularity_metrics["Coverage"] = popularity_coverage * 100

popularity_metrics_df = pd.DataFrame([popularity_metrics])

# Conserver les résultats Content-Based déjà enregistrés
results_path = "../data/processed/evaluation_results.csv"

if os.path.exists(results_path):
    all_results = pd.read_csv(results_path)
    all_results = all_results[
        all_results["Model"] != "Popularity"
    ]
    all_results = pd.concat(
        [all_results, popularity_metrics_df],
        ignore_index=True
    )
else:
    all_results = popularity_metrics_df

all_results.to_csv(results_path, index=False)

print("Résultats Popularity sauvegardés.")
display(all_results.round(4))

Résultats Popularity sauvegardés.


,Model,Precision@10,Recall@10,NDCG@10,Coverage,Evaluated impressions,Sample impressions
0,Content-Based,0.0831,0.6250,0.3836,0.0654,965,1000
1,Popularity,0.0693,0.5481,0.3039,0.8308,1000,1000
